# NER Token Indexing: From Trained Models to Index Creation

This notebook demonstrates how to:
1. Load a trained NER model (BERT fine-tuned on CoNLL 2003)
2. Extract token embeddings from the dataset
3. Build searchable indexes using different backends (Torch, Annoy, FAISS)
4. Query the index to find nearest neighbor tokens

**Key Components:**
- `src/indexing/base.py` - Main facade with `build_index()` and `query_index()`
- `src/indexing/ner_token_index.py` - NER-specific token extraction logic
- Three backends: torch (in-memory), annoy (disk), faiss (optimized L2)

In [1]:
# Setup: Import paths and dependencies
import sys
import os
from pathlib import Path

# Find repository root (folder containing src/), regardless of notebook CWD
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src").exists()), None)

if project_root is None:
    raise RuntimeError(f"Could not locate project root from cwd={cwd}. Expected a parent containing 'src/'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Working directory: {cwd}")
print(f"src exists: {(project_root / 'src').exists()}")

Project root: D:\THESIS\RandomizedSmothingmanifold
Working directory: D:\THESIS\RandomizedSmothingmanifold\notebooks
src exists: True


## Step 1: Load Trained NER Model and DataLoader

Load the fine-tuned BERT model from  trained checkpoint and set up the CoNLL 2003 dataset loader.

In [ ]:
# Import core libraries
import sys
import subprocess
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import json
from pprint import pprint

# Import from project
from src.indexing import (
    build_index,
    query_index,
    neighbor_vectors,
    NeighborIndex,
    save_faiss_index,
    save_annoy_index,
    load_token_index_artifacts,
)
from src.smoothing import fit_local_pca, whiten, unwhiten
from transformers import AutoTokenizer

print("All imports successful")

All imports successful


## Step 2: Extract Labeled NER Token Embeddings

Build the index from contextual token vectors taken from the train split, keeping only positions with real NER labels.

This excludes:
- special tokens like `[CLS]`, `[SEP]`, `[PAD]`
- subword positions masked with label `-100`
- any unlabeled positions that should not affect NER neighborhoods

In [3]:
# Load trained NER model config
config_path = Path("../output/ner_conll2003_bert/ner_bert_conll2003_finetune/resolved_config.yaml")
model_checkpoint = Path("../output/ner_conll2003_bert/ner_bert_conll2003_finetune/model.pt")

import yaml
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Model Config:")
pprint(config)
print(f"\nModel checkpoint exists: {model_checkpoint.exists()}")
print(f"Config path exists: {config_path.exists()}")

Model Config:
{'certification': {'abstain_label': -1,
                   'alpha': 0.001,
                   'enabled': False,
                   'n': 512,
                   'n0': 64},
 'dataloader': {'batch_size': 16, 'shuffle_train': True},
 'dataset': {'label_field': 'ner_tags',
             'max_length': 192,
             'name': 'conll2003',
             'num_workers': 2,
             'split_test': 'test',
             'split_train': 'train',
             'split_val': 'validation',
             'text_field': 'tokens'},
 'eval': {'max_batches': None},
 'experiment_name': 'ner_bert_conll2003_finetune',
 'model': {'dropout': 0.1, 'encoder_name': 'bert-base-uncased'},
 'output_dir': 'output/ner_conll2003_bert',
 'smoothing': {'enabled': False,
               'eps_eig': 1e-06,
               'index_backend': 'torch',
               'index_metric': 'euclidean',
               'index_n_trees': 20,
               'index_path': None,
               'knn_k': 0,
               'layer_index':

In [4]:
# Load model and tokenizer (match checkpoint architecture)
from src.models.transformer.ner.model import TransformerNER

model_name = config["model"]["encoder_name"]  # e.g. bert-base-uncased
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading model: {model_name} on device: {device}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load fine-tuned weights
checkpoint = torch.load(model_checkpoint, map_location=device)
state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))

# Infer number of labels directly from checkpoint classifier head
if "classifier.weight" not in state_dict:
    raise KeyError("Expected 'classifier.weight' in checkpoint state_dict, but it was not found.")

num_labels = int(state_dict["classifier.weight"].shape[0])
id2label = {i: f"LABEL_{i}" for i in range(num_labels)}
label2id = {v: k for k, v in id2label.items()}

dropout = config.get("model", {}).get("dropout", 0.1)
model = TransformerNER(
    encoder_name=model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dropout=dropout,
).to(device)

missing, unexpected = model.load_state_dict(state_dict, strict=False)
model.eval()

print(f"Model loaded from {model_checkpoint}")
print(f"num_labels: {num_labels}")
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")

Loading model: bert-base-uncased on device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded from ..\output\ner_conll2003_bert\ner_bert_conll2003_finetune\model.pt
num_labels: 9
Missing keys: 0 | Unexpected keys: 0


In [5]:
# Build/load labeled token artifacts outside the notebook kernel
if "config_path" not in globals() or "model_checkpoint" not in globals():
    raise RuntimeError("Run the config cell first so config_path and model_checkpoint are available.")

artifacts_dir = project_root / "output" / "ner_conll2003_bert" / "ner_bert_conll2003_finetune" / "token_index_train"
vectors_path = artifacts_dir / "token_vectors.npz"
meta_path = artifacts_dir / "token_metadata.json"
label_map_path = artifacts_dir / "label_map.json"

if not (vectors_path.exists() and meta_path.exists() and label_map_path.exists()):
    print("Token artifacts not found. Building them in an external Python process...")
    cmd = [
        sys.executable,
        "-m",
        "src.indexing.build_ner_token_artifacts",
        "--config",
        str(config_path.resolve()),
        "--checkpoint",
        str(model_checkpoint.resolve()),
        "--out-dir",
        str(artifacts_dir.resolve()),
        "--split",
        "train",
    ]
    subprocess.run(cmd, check=True, cwd=str(project_root))
else:
    print(f"Using existing token artifacts in: {artifacts_dir}")

with open(label_map_path) as f:
    label_map = json.load(f)

label_names = [label_map["id2label"][str(i)] for i in range(len(label_map["id2label"]))]
print(f"Artifacts directory: {artifacts_dir}")
print(f"NER labels: {label_names}")

Using existing token artifacts in: D:\THESIS\RandomizedSmothingmanifold\output\ner_conll2003_bert\ner_bert_conll2003_finetune\token_index_train
Artifacts directory: D:\THESIS\RandomizedSmothingmanifold\output\ner_conll2003_bert\ner_bert_conll2003_finetune\token_index_train
NER labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


## Step 3: Build Indexes with Different Backends

Now we'll create searchable indexes using three different backends:
1. **Torch**: Fast, in-memory, best for small/medium datasets
2. **Annoy**: Disk-saveable, good for large-scale experiments
3. **FAISS**: Optimized GPU operations, best for very large datasets

In [6]:
# Load contextual vectors for labeled token positions only
if "artifacts_dir" not in globals():
    raise RuntimeError("Run the artifact-build cell first.")

token_vectors, token_texts, label_ids = load_token_index_artifacts(artifacts_dir)

all_embeddings = token_vectors
index_tokens = token_texts
index_labels = [label_map["id2label"][str(int(label_id))] for label_id in label_ids]

total_tokens = all_embeddings.shape[0]
print(f"Total labeled token vectors extracted: {all_embeddings.shape}")
print(f"  Shape: (n_labeled_tokens={all_embeddings.shape[0]}, embedding_dim={all_embeddings.shape[1]})")
print(f"  Dtype: {all_embeddings.dtype}")
print(f"  Memory: {all_embeddings.nbytes / 1e6:.2f} MB")
print(f"  Example token/label: {index_tokens[0]} / {index_labels[0]}")

Total labeled token vectors extracted: (203621, 768)
  Shape: (n_labeled_tokens=203621, embedding_dim=768)
  Dtype: float32
  Memory: 625.52 MB
  Example token/label: eu / B-ORG


In [7]:
# Quick peek at extracted labeled tokens and labels
list(zip(index_tokens[:20], index_labels[:20]))

[('eu', 'B-ORG'),
 ('rejects', 'O'),
 ('german', 'B-MISC'),
 ('call', 'O'),
 ('to', 'O'),
 ('boycott', 'O'),
 ('british', 'B-MISC'),
 ('lamb', 'O'),
 ('.', 'O'),
 ('peter', 'B-PER'),
 ('blackburn', 'I-PER'),
 ('brussels', 'B-LOC'),
 ('1996', 'O'),
 ('the', 'O'),
 ('european', 'B-ORG'),
 ('commission', 'I-ORG'),
 ('said', 'O'),
 ('on', 'O'),
 ('thursday', 'O'),
 ('it', 'O')]

In [8]:
# Quick peek at extracted vector matrix
all_embeddings.shape

(203621, 768)

In [9]:
# Build FAISS index (optimized, GPU-accelerated)
print("\n" + "=" * 60)
print("Building FAISS index...")
print("=" * 60)

faiss_index = build_index(
    vectors=all_embeddings,
    backend='faiss',
    metric='euclidean'
)

print(f"✓ FAISS index created:")
print(f"  Backend: {faiss_index.backend}")
print(f"  Dimension: {faiss_index.dim}")
print(f"  Metric: {faiss_index.metric}")
print(f"  Index type: {type(faiss_index.index).__name__}")

# Save FAISS index via project indexing API
faiss_save_path = Path("notebooks/celeba_ner_tokens_faiss.faiss")
save_faiss_index(faiss_index, str(faiss_save_path))
print(f"✓ FAISS index saved to: {faiss_save_path}")

# Build ANNOY index (tree-based, disk-saveable)
print("\n" + "=" * 60)
print("Building ANNOY index...")
print("=" * 60)

annoy_index = build_index(
    vectors=all_embeddings,
    backend='annoy',
    metric='euclidean'
)

print(f"✓ Annoy index created:")
print(f"  Backend: {annoy_index.backend}")
print(f"  Dimension: {annoy_index.dim}")
print(f"  Metric: {annoy_index.metric}")
print(f"  Index type: {type(annoy_index.index).__name__}")

# Save Annoy index via project indexing API
annoy_save_path = Path("notebooks/celeba_ner_tokens_annoy.ann")
save_annoy_index(annoy_index, str(annoy_save_path))
print(f"✓ Annoy index saved to: {annoy_save_path}")

# Build TORCH index (in-memory L2 distance)
print("=" * 60)
print("Building TORCH index...")
print("=" * 60)

torch_index = build_index(
    vectors=all_embeddings,
    backend='torch',
    metric='l2'
)

print(f"✓ Torch index created:")
print(f"  Backend: {torch_index.backend}")
print(f"  Dimension: {torch_index.dim}")
print(f"  Metric: {torch_index.metric}")
print(f"  Index type: {type(torch_index.index)}")
print(f"  Index shape: {torch_index.index.shape if hasattr(torch_index.index, 'shape') else 'N/A'}")


Building FAISS index...
✓ FAISS index created:
  Backend: faiss
  Dimension: 768
  Metric: euclidean
  Index type: IndexFlatL2
✓ FAISS index saved to: notebooks\celeba_ner_tokens_faiss.faiss

Building ANNOY index...
✓ Annoy index created:
  Backend: annoy
  Dimension: 768
  Metric: euclidean
  Index type: Annoy
✓ Annoy index saved to: notebooks\celeba_ner_tokens_annoy.ann
Building TORCH index...
✓ Torch index created:
  Backend: torch
  Dimension: 768
  Metric: euclidean
  Index type: <class 'NoneType'>
  Index shape: N/A


In [12]:
# Step 4: Query each index and compare nearest neighbors
# query_index(...) returns neighbor ID arrays, so we compute distances from embeddings.

torch.manual_seed(42)
np.random.seed(42)

def l2_distances(anchor_vec: np.ndarray, neighbor_ids: np.ndarray, embedding_matrix: np.ndarray) -> np.ndarray:
    rows = embedding_matrix[neighbor_ids]
    return np.linalg.norm(rows - anchor_vec.reshape(1, -1), axis=1)

# Pick 3 random token indices to query
k = 5
test_indices = np.random.choice(all_embeddings.shape[0], 20, replace=False)
query_results = {}

print("=" * 70)
print(f"Querying all backends with k={k}")
print("=" * 70)

for idx in test_indices:
    idx = int(idx)
    query_vec = all_embeddings[idx]
    query_tok = index_tokens[idx]

    # IDs from each backend
    ids_torch = query_index(torch_index, k=k, vector=query_vec)
    ids_annoy = query_index(annoy_index, k=k, vector=query_vec)
    ids_faiss = query_index(faiss_index, k=k, vector=query_vec)

    # Distances computed against the shared embedding table
    d_torch = l2_distances(query_vec, ids_torch, all_embeddings)
    d_annoy = l2_distances(query_vec, ids_annoy, all_embeddings)
    d_faiss = l2_distances(query_vec, ids_faiss, all_embeddings)

    query_results[idx] = {
        "query_token": query_tok,
        "torch": {"ids": ids_torch, "dist": d_torch},
        "annoy": {"ids": ids_annoy, "dist": d_annoy},
        "faiss": {"ids": ids_faiss, "dist": d_faiss},
    }

    print(f"\nQuery idx={idx}, token='{query_tok}'")
    print("-" * 70)
    for name, ids, dists in [
        ("TORCH", ids_torch, d_torch),
        ("ANNOY", ids_annoy, d_annoy),
        ("FAISS", ids_faiss, d_faiss),
    ]:
        toks = [index_tokens[int(i)] for i in ids]
        print(f"{name:>6} ids: {ids.tolist()}")
        print(f"{name:>6} tok: {toks}")
        print(f"{name:>6} l2 : {[round(float(x), 6) for x in dists]}")

print("\nDone. query_results dict is available for Step 5.")

Querying all backends with k=5

Query idx=126963, token='u'
----------------------------------------------------------------------
 TORCH ids: [126963, 40167, 181727, 188826, 127598]
 TORCH tok: ['u', 'u', 'u', 'u', 'u']
 TORCH l2 : [0.0, 1.368587, 1.974023, 2.221395, 2.224812]
 ANNOY ids: [126963, 40167, 181727, 188826, 127598]
 ANNOY tok: ['u', 'u', 'u', 'u', 'u']
 ANNOY l2 : [0.0, 1.368587, 1.974023, 2.221395, 2.224812]
 FAISS ids: [126963, 40167, 181727, 188826, 127598]
 FAISS tok: ['u', 'u', 'u', 'u', 'u']
 FAISS l2 : [0.0, 1.368587, 1.974023, 2.221395, 2.224812]

Query idx=182817, token='.'
----------------------------------------------------------------------
 TORCH ids: [182817, 135971, 182513, 82382, 124417]
 TORCH tok: ['.', '.', '.', '.', '.']
 TORCH l2 : [0.0, 1.664024, 1.707774, 1.721355, 1.722681]
 ANNOY ids: [182817, 135971, 182513, 82382, 124417]
 ANNOY tok: ['.', '.', '.', '.', '.']
 ANNOY l2 : [0.0, 1.664024, 1.707774, 1.721355, 1.722681]
 FAISS ids: [182817, 135971, 

In [13]:
# Step 5: Retrieve neighbor vectors and verify distance consistency
print("=" * 70)
print("Retrieving neighbor vectors for analysis")
print("=" * 70)

if "query_results" not in globals() or not query_results:
    raise RuntimeError("Run Step 4 first so query_results is populated.")

query_idx = int(test_indices[0])
query_vec = all_embeddings[query_idx]
query_tok = index_tokens[query_idx]
k = 5

# Use API helper for torch vectors; use ids for annoy/faiss
torch_neighbor_vecs = neighbor_vectors(torch_index, k=k, vector=query_vec)
annoy_ids = query_results[query_idx]["annoy"]["ids"]
faiss_ids = query_results[query_idx]["faiss"]["ids"]
annoy_neighbor_vecs = all_embeddings[annoy_ids]
faiss_neighbor_vecs = all_embeddings[faiss_ids]

# Compute L2 distances explicitly
l2_torch = np.linalg.norm(torch_neighbor_vecs - query_vec.reshape(1, -1), axis=1)
l2_annoy = np.linalg.norm(annoy_neighbor_vecs - query_vec.reshape(1, -1), axis=1)
l2_faiss = np.linalg.norm(faiss_neighbor_vecs - query_vec.reshape(1, -1), axis=1)

print(f"Query idx={query_idx}, token='{query_tok}', vector shape={query_vec.shape}")
print("\nTop-k tokens by backend:")
print(f"  TORCH: {[index_tokens[int(i)] for i in query_results[query_idx]['torch']['ids']]}")
print(f"  ANNOY: {[index_tokens[int(i)] for i in annoy_ids]}")
print(f"  FAISS: {[index_tokens[int(i)] for i in faiss_ids]}")

print("\nDistance check (first 5):")
print(f"  TORCH: {[round(float(x), 6) for x in l2_torch]}")
print(f"  ANNOY: {[round(float(x), 6) for x in l2_annoy]}")
print(f"  FAISS: {[round(float(x), 6) for x in l2_faiss]}")

print("\nSanity checks:")
print(f"  torch vecs shape: {torch_neighbor_vecs.shape}")
print(f"  annoy vecs shape: {annoy_neighbor_vecs.shape}")
print(f"  faiss vecs shape: {faiss_neighbor_vecs.shape}")
print(f"  Self-neighbor present (torch): {query_idx in set(query_results[query_idx]['torch']['ids'].tolist())}")

Retrieving neighbor vectors for analysis
Query idx=126963, token='u', vector shape=(768,)

Top-k tokens by backend:
  TORCH: ['u', 'u', 'u', 'u', 'u']
  ANNOY: ['u', 'u', 'u', 'u', 'u']
  FAISS: ['u', 'u', 'u', 'u', 'u']

Distance check (first 5):
  TORCH: [0.0, 1.368587, 1.974023, 2.221395, 2.224812]
  ANNOY: [0.0, 1.368587, 1.974023, 2.221395, 2.224812]
  FAISS: [0.0, 1.368587, 1.974023, 2.221395, 2.224812]

Sanity checks:
  torch vecs shape: (5, 768)
  annoy vecs shape: (5, 768)
  faiss vecs shape: (5, 768)
  Self-neighbor present (torch): True


## Step 6: PCA + Whiten/Unwhiten Diagnostics on CoNLL Tokens

This section tests local PCA smoothing on sampled labeled token vectors and reports:
- original row (first dimensions)
- noisy row (noise added in whitened PCA space)
- PCA reconstructed row
- original NER label
- noisy-input nearest-neighbor labels and classifier predictions

In [14]:
# Step 6A: Utility helpers for PCA diagnostic
np.random.seed(123)
torch.manual_seed(123)

id2label_map = label_map["id2label"] if "label_map" in globals() else {str(i): f"LABEL_{i}" for i in range(num_labels)}

def _pred_label_from_vector(vec: np.ndarray) -> str:
    """Run NER classifier head directly on a single hidden-state vector."""
    with torch.no_grad():
        x = torch.as_tensor(vec, dtype=torch.float32, device=device).unsqueeze(0)
        logits = model.classifier(model.dropout(x))
        pred_id = int(torch.argmax(logits, dim=-1).item())
    return id2label_map[str(pred_id)]

def _label_counts(labels: list[str]) -> dict[str, int]:
    out = {}
    for lab in labels:
        out[lab] = out.get(lab, 0) + 1
    return out

print("Helpers ready.")

Helpers ready.


In [ ]:
# Step 6B: Run PCA -> whiten -> noisy -> unwhiten on sampled CoNLL token vectors
# Self-contained import so this cell runs even after kernel restart.
from src.smoothing import fit_local_pca, whiten, unwhiten

if "all_embeddings" not in globals() or "index_labels" not in globals():
    raise RuntimeError("Run Step 3 first so embeddings/labels are loaded.")

n_samples = 5
k_local = 64
sigma = 0.20

diag_indices = np.random.choice(all_embeddings.shape[0], n_samples, replace=False)
print(f"Running diagnostics on {n_samples} samples (k_local={k_local}, sigma={sigma})")

for idx in diag_indices:
    idx = int(idx)
    anchor = all_embeddings[idx].astype(np.float32)
    token = index_tokens[idx]
    gold_label = index_labels[idx]

    # Local neighborhood for PCA fit
    neigh_ids = query_index(torch_index, k=k_local, vector=anchor)
    neigh_vecs = all_embeddings[neigh_ids]

    # PCA + whiten/unwhiten pipeline
    pca = fit_local_pca(neigh_vecs, eps_eig=1e-6)
    z_anchor = whiten(anchor, pca)
    z_noise = np.random.randn(*z_anchor.shape).astype(np.float32) * sigma
    noisy_vec = unwhiten(z_anchor + z_noise, pca).astype(np.float32)
    recon_vec = unwhiten(z_anchor, pca).astype(np.float32)

    # Label probes for noisy input
    noisy_nn_ids = query_index(torch_index, k=5, vector=noisy_vec)
    noisy_nn_labels = [index_labels[int(i)] for i in noisy_nn_ids]
    noisy_nn_tokens = [index_tokens[int(i)] for i in noisy_nn_ids]

    pred_orig = _pred_label_from_vector(anchor)
    pred_noisy = _pred_label_from_vector(noisy_vec)
    pred_recon = _pred_label_from_vector(recon_vec)

    rec_err = float(np.linalg.norm(anchor - recon_vec))
    noise_delta = float(np.linalg.norm(anchor - noisy_vec))

    print("\n" + "=" * 90)
    print(f"idx={idx} | token='{token}' | gold_label={gold_label}")
    print("-" * 90)
    print(f"orig_row[:12] : {np.round(anchor[:12], 4).tolist()}")
    print(f"noisy_row[:12]: {np.round(noisy_vec[:12], 4).tolist()}")
    print(f"recon_row[:12]: {np.round(recon_vec[:12], 4).tolist()}")
    print(f"||orig-recon||_2 = {rec_err:.8f}  (should be near 0)")
    print(f"||orig-noisy||_2 = {noise_delta:.6f}")
    print(f"pred_labels -> original: {pred_orig}, noisy: {pred_noisy}, recon: {pred_recon}")
    print(f"noisy_input_neighbor_tokens: {noisy_nn_tokens}")
    print(f"noisy_input_neighbor_labels: {noisy_nn_labels}")
    print(f"noisy_label_histogram: {_label_counts(noisy_nn_labels)}")

Running diagnostics on 50 samples (k_local=64, sigma=0.2)

idx=25566 | token='15' | gold_label=O
------------------------------------------------------------------------------------------
orig_row[:12] : [-0.5863999724388123, -0.8230999708175659, 0.5968000292778015, 0.3716999888420105, -0.9679999947547913, -0.015799999237060547, -1.2791999578475952, 0.07800000160932541, -1.2032999992370605, -0.7483000159263611, 0.5954999923706055, 1.0055999755859375]
noisy_row[:12]: [-0.5949000120162964, -0.8184000253677368, 0.592199981212616, 0.3488999903202057, -0.9749000072479248, -0.007699999958276749, -1.2691999673843384, 0.07769999653100967, -1.2167999744415283, -0.7501999735832214, 0.5924999713897705, 1.0027999877929688]
recon_row[:12]: [-0.5863999724388123, -0.8230999708175659, 0.5968000292778015, 0.3716999888420105, -0.9679999947547913, -0.015799999237060547, -1.2791999578475952, 0.07800000160932541, -1.2032999992370605, -0.7483000159263611, 0.5954999923706055, 1.0055999755859375]
||orig-recon